# 03 · Silver → Gold (Star Schema) — Contoso Retail 360

Builds the **Gold** dimensional model: `DimDate`, `DimProduct`, `DimStore`,
`DimCustomer`, `DimChannel`, and `FactSales`. Create a `gold` schema first.
Surrogate keys are generated here; the fact joins Silver to dimension keys.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER, GOLD = "silver", "gold"

def with_skey(df, name):
    """Add a stable surrogate key column."""
    w = Window.orderBy(F.monotonically_increasing_id())
    return df.withColumn(name, F.row_number().over(w))

In [ ]:
# ---------------- DimDate ----------------
sales = spark.read.table(f"{SILVER}.sales")
bounds = sales.agg(F.min("OrderDate").alias("mn"), F.max("OrderDate").alias("mx")).collect()[0]
date_df = (spark.sql(
    f"SELECT explode(sequence(to_date('{bounds.mn}'), to_date('{bounds.mx}'), interval 1 day)) AS Date"))
dim_date = (date_df
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("Day", F.dayofmonth("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("Year", F.year("Date"))
    .withColumn("DayOfWeek", F.date_format("Date", "EEEE"))
    .withColumn("IsWeekend", F.dayofweek("Date").isin([1, 7])))
dim_date.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.DimDate")
print("DimDate:", dim_date.count())

In [ ]:
# ---------------- DimProduct ----------------
dim_product = with_skey(spark.read.table(f"{SILVER}.products"), "ProductKey")
dim_product.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.DimProduct")

# ---------------- DimStore ----------------
dim_store = with_skey(spark.read.table(f"{SILVER}.stores"), "StoreKey")
dim_store.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.DimStore")

# ---------------- DimCustomer ----------------
dim_customer = with_skey(spark.read.table(f"{SILVER}.customers"), "CustomerKey")
dim_customer.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.DimCustomer")

# ---------------- DimChannel ----------------
dim_channel = spark.createDataFrame(
    [(1, "Store"), (2, "Online")], ["ChannelKey", "Channel"])
dim_channel.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.DimChannel")
print("dimensions written")

In [ ]:
# ---------------- FactSales ----------------
sales   = spark.read.table(f"{SILVER}.sales")
dprod   = spark.read.table(f"{GOLD}.DimProduct").select("ProductKey", "ProductId", "UnitCost")
dstore  = spark.read.table(f"{GOLD}.DimStore").select("StoreKey", "StoreId")
dcust   = spark.read.table(f"{GOLD}.DimCustomer").select("CustomerKey", "CustomerId")
dchan   = spark.read.table(f"{GOLD}.DimChannel")

fact = (sales
    .join(dprod,  "ProductId",  "left")
    .join(dstore, "StoreId",    "left")
    .join(dcust,  "CustomerId", "left")
    .join(dchan,  "Channel",    "left")
    .withColumn("DateKey", F.date_format("OrderDate", "yyyyMMdd").cast("int"))
    .withColumn("SalesAmount",
        (F.col("Quantity") * F.col("UnitPrice") - F.col("DiscountAmount")).cast("decimal(18,2)"))
    .select(
        F.monotonically_increasing_id().alias("SalesKey"),
        "DateKey", "ProductKey", "StoreKey", "CustomerKey", "ChannelKey",
        "OrderNumber", "Quantity", "UnitPrice", "DiscountAmount", "SalesAmount",
        F.col("UnitCost")))
fact.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.FactSales")
print("FactSales:", fact.count())

In [ ]:
# ---------------- Validation: no orphan keys ----------------
fact = spark.read.table(f"{GOLD}.FactSales")
orphans = fact.filter(
    F.col("ProductKey").isNull() | F.col("StoreKey").isNull() |
    F.col("CustomerKey").isNull() | F.col("ChannelKey").isNull()).count()
print("orphan fact rows (should be 0):", orphans)
display(fact.limit(5))